# 🌊 Seismic Data Analysis & Attribute Computation
### Full Workflow for 2-D SEG-Y Data (227 samples × 112,875 traces)

---

**Notebook overview**

| Section | Description |
|---------|-------------|
| 1 | Environment setup & library imports |
| 2 | SEG-Y loading & metadata inspection |
| 3 | Data quality checks & statistics |
| 4 | Raw trace visualisation |
| 5 | Seismic section (wiggle + variable-density) |
| 6 | Amplitude envelope (Instantaneous Amplitude) |
| 7 | Instantaneous Phase & Frequency |
| 8 | Spectral (frequency-domain) analysis |
| 9 | Semblance / Coherence |
| 10 | RMS Amplitude (sliding window) |
| 11 | Acoustic Impedance proxy & Reflection Coefficient |
| 12 | Sweetness attribute |
| 13 | Dominant frequency map |
| 14 | Full attribute gallery |

> **Coordinate note:** The SEG-Y file has no geometry headers — traces are numbered sequentially 0 … N-1.  
> All spatial axes in this notebook represent **trace index**, not X/Y/CDP coordinates.


## 1 · Environment Setup & Library Imports

### Theory
We rely on four key libraries:

* **`segyio`** — reads SEG-Y Rev-1/2 files without requiring geometry.  
* **`NumPy`** — vectorised array maths (FFT, convolution, rolling windows).  
* **`SciPy`** — Hilbert transform (analytic signal → instantaneous attributes), filters.  
* **`Matplotlib / Seaborn`** — publication-quality plotting.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import seaborn as sns
from scipy.signal import hilbert, welch, butter, filtfilt, spectrogram
from scipy.ndimage import uniform_filter1d
import segyio
import os, time

# ── Plotting defaults ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'      : 120,
    'axes.titlesize'  : 12,
    'axes.labelsize'  : 10,
    'xtick.labelsize' : 8,
    'ytick.labelsize' : 8,
    'font.family'     : 'DejaVu Sans',
})
CMAP_SEIS  = 'seismic'
CMAP_AMP   = 'plasma'
CMAP_PHASE = 'hsv'
CMAP_FREQ  = 'jet'

print("✅ All libraries imported successfully.")
print(f"   numpy  {np.__version__}  |  segyio {segyio.__version__}  |  scipy loaded")


## 2 · SEG-Y Loading & Metadata Inspection

### Theory
SEG-Y is the industry-standard binary format for seismic data.  
Each **trace** stores:
- A 240-byte **binary header** (shot/receiver coordinates, sample interval, etc.)
- A 1-D amplitude array of length `n_samples`

When geometry headers are absent (as here), we load the file in **"ignore geometry"** mode
and treat every trace as an independent 1-D time series.

Key parameters:
| Symbol | Value | Meaning |
|--------|-------|---------|
| `dt`   | from header | Sample interval (µs → ms) |
| `ns`   | 227 | Samples per trace |
| `nt`   | 112 875 | Number of traces |
| `T`    | ns × dt | Total record time |


In [ ]:
# ── ▶▶  UPDATE THIS PATH TO YOUR SEG-Y FILE  ◀◀ ──────────────────────────────
SEGY_PATH = "your_file.segy"          # e.g. r"C:/data/survey.segy"
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.exists(SEGY_PATH):
    # ── Demo / synthetic data when no real file is present ────────────────────
    print("⚠️  SEG-Y file not found — generating synthetic data for demonstration.")
    np.random.seed(42)
    N_SAMPLES, N_TRACES = 227, 112875
    dt_ms = 2.0                                      # 2 ms sample interval

    # Ricker wavelet (dominant freq 35 Hz)
    def ricker(f, dt, ns):
        t  = (np.arange(ns) - ns//2) * dt/1000
        pi2 = (np.pi * f * t) ** 2
        return (1 - 2*pi2) * np.exp(-pi2)

    wavelet = ricker(35, dt_ms, N_SAMPLES)

    # Reflectivity series — 3 main reflectors + noise
    ref = np.zeros(N_SAMPLES)
    ref[[40, 100, 160]] = [0.8, -0.5, 0.3]

    from scipy.signal import fftconvolve
    base_trace = fftconvolve(ref, wavelet, mode='same')

    # Add spatial variation (gentle lateral velocity trend)
    traces_list = []
    for i in range(N_TRACES):
        shift   = int(5 * np.sin(2 * np.pi * i / N_TRACES))
        shifted = np.roll(base_trace, shift)
        noise   = np.random.randn(N_SAMPLES) * 0.05
        traces_list.append(shifted + noise)

    data = np.array(traces_list).T          # shape (n_samples, n_traces)
    print(f"✅ Synthetic data created  |  shape {data.shape}  |  dt = {dt_ms} ms")

else:
    t0 = time.time()
    with segyio.open(SEGY_PATH, ignore_geometry=True) as f:
        dt_ms   = segyio.tools.dt(f) / 1000.0          # µs → ms
        N_SAMPLES = f.samples.size
        N_TRACES  = f.tracecount
        data      = segyio.tools.collect(f.trace[:])   # (n_traces, n_samples)
        data      = data.T                              # → (n_samples, n_traces)
    print(f"✅ Loaded  |  {time.time()-t0:.1f}s  |  shape {data.shape}  |  dt = {dt_ms} ms")

# ── Derived parameters ─────────────────────────────────────────────────────────
time_axis  = np.arange(N_SAMPLES) * dt_ms      # ms
trace_axis = np.arange(N_TRACES)
fs         = 1000.0 / dt_ms                    # Hz  (sampling frequency)
nyquist    = fs / 2.0

print(f"   Samples   : {N_SAMPLES}")
print(f"   Traces    : {N_TRACES}")
print(f"   dt        : {dt_ms} ms  →  fs = {fs:.0f} Hz  |  Nyquist = {nyquist:.0f} Hz")
print(f"   Record length : {time_axis[-1]:.1f} ms")


## 3 · Data Quality Checks & Statistics

### Theory
Before computing any attribute we must understand the **amplitude distribution**.

- **Clipping** → histogram with sharp cutoffs → normalise or clip before attributes  
- **Outlier traces** → dead traces (all zeros) or noisy traces  
- **Dynamic range** → ratio of max to RMS amplitude; > 60 dB is typical for quality data


In [ ]:
# ── Per-trace RMS ─────────────────────────────────────────────────────────────
rms_per_trace = np.sqrt(np.mean(data**2, axis=0))

# ── Global statistics ─────────────────────────────────────────────────────────
g_min, g_max  = data.min(), data.max()
g_mean        = data.mean()
g_std         = data.std()
g_rms         = np.sqrt(np.mean(data**2))
dynamic_range = 20 * np.log10(g_max / (g_rms + 1e-10))

dead_traces   = np.sum(rms_per_trace == 0)

print("=" * 50)
print("  GLOBAL AMPLITUDE STATISTICS")
print("=" * 50)
print(f"  Min            : {g_min:+.4f}")
print(f"  Max            : {g_max:+.4f}")
print(f"  Mean           : {g_mean:+.6f}")
print(f"  Std dev        : {g_std:.4f}")
print(f"  RMS            : {g_rms:.4f}")
print(f"  Dynamic range  : {dynamic_range:.1f} dB")
print(f"  Dead traces    : {dead_traces}")
print("=" * 50)

# ── Figure: amplitude histogram + per-trace RMS ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(data.ravel(), bins=300, color='steelblue', edgecolor='none', alpha=0.85)
axes[0].set_xlabel("Amplitude"); axes[0].set_ylabel("Count")
axes[0].set_title("Global Amplitude Histogram")
axes[0].axvline(g_rms,  color='red',    lw=1.5, label=f'RMS={g_rms:.3f}')
axes[0].axvline(-g_rms, color='red',    lw=1.5)
axes[0].axvline(g_mean, color='orange', lw=1.5, label=f'Mean={g_mean:.4f}')
axes[0].legend(fontsize=8)

step = max(1, N_TRACES // 5000)          # decimate for speed
axes[1].plot(trace_axis[::step], rms_per_trace[::step], lw=0.4, color='navy')
axes[1].set_xlabel("Trace index"); axes[1].set_ylabel("RMS amplitude")
axes[1].set_title("Per-Trace RMS  (identifies dead / noisy traces)")
axes[1].axhline(rms_per_trace.mean(), color='red', lw=1, ls='--', label='Mean RMS')
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()
print("✅ Quality check complete.")


## 4 · Raw Trace Visualisation

### Theory
We plot a **wiggle display** of a handful of individual traces.  
This is the most direct look at the seismic waveform — each trace is a time-series
of reflected acoustic energy.

Key waveform features to observe:
- **Zero crossings** → reflector boundaries
- **Peak polarity** → acoustic impedance contrast direction
- **Wavelet shape** → bandwidth, phase, ringing


In [ ]:
N_WIGGLE = 10          # how many traces to show
step_w   = N_TRACES // N_WIGGLE

fig, ax = plt.subplots(figsize=(14, 6))
for k, ti in enumerate(range(0, N_TRACES, step_w)):
    tr  = data[:, ti]
    tr_n = tr / (np.abs(tr).max() + 1e-10)       # normalise to [-1, 1]
    ax.plot(tr_n * 0.8 + k, time_axis, lw=0.8, color='black')
    # shade positive lobes
    ax.fill_betweenx(time_axis, k, tr_n * 0.8 + k,
                     where=(tr_n > 0), color='red',  alpha=0.5)
    ax.fill_betweenx(time_axis, k, tr_n * 0.8 + k,
                     where=(tr_n < 0), color='blue', alpha=0.3)

ax.set_xlim(-0.5, N_WIGGLE)
ax.set_xticks(range(N_WIGGLE))
ax.set_xticklabels([f"T{i*step_w}" for i in range(N_WIGGLE)], rotation=30, ha='right')
ax.invert_yaxis()
ax.set_xlabel("Trace index")
ax.set_ylabel("Two-Way Time (ms)")
ax.set_title(f"Wiggle Display — {N_WIGGLE} Equally-Spaced Traces")
ax.grid(axis='y', lw=0.3, alpha=0.5)
plt.tight_layout(); plt.show()


## 5 · Seismic Section — Variable-Density Image (VDI)

### Theory
A **seismic section** is a 2-D image where:
- **X-axis** = trace index (space / CDP)
- **Y-axis** = two-way travel time (TWT, ms) — increases downward (geological convention)
- **Colour** = amplitude polarity & strength

The **variable-density display** (colour-coded) is the standard interpretation image.  
Red/blue diverging colourmaps centre on zero amplitude (white = nodal / zero-crossing).

We clip the colour scale at ±2σ to prevent a few outliers from washing out structure.


In [ ]:
# ── Sub-section for display (full 112875 traces → decimate for performance) ──
DISP_TRACES = 2000                          # traces to display
t_start, t_end = 0, N_TRACES               # change to zoom a region
idx = np.linspace(t_start, t_end-1, DISP_TRACES, dtype=int)
sec = data[:, idx]

clip = 2.0 * sec.std()

fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)

# Variable-density
im = axes[0].imshow(sec, aspect='auto', cmap=CMAP_SEIS,
                    vmin=-clip, vmax=clip,
                    extent=[idx[0], idx[-1], time_axis[-1], time_axis[0]])
plt.colorbar(im, ax=axes[0], label='Amplitude', shrink=0.8)
axes[0].set_xlabel("Trace index"); axes[0].set_ylabel("TWT (ms)")
axes[0].set_title("Seismic Section — Variable Density Image")

# Greyscale positive-only
im2 = axes[1].imshow(np.abs(sec), aspect='auto', cmap='Greys',
                     vmin=0, vmax=clip,
                     extent=[idx[0], idx[-1], time_axis[-1], time_axis[0]])
plt.colorbar(im2, ax=axes[1], label='|Amplitude|', shrink=0.8)
axes[1].set_xlabel("Trace index")
axes[1].set_title("Seismic Section — Greyscale (|Amplitude|)")

plt.suptitle("Full Seismic Section  (2 000 representative traces)", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()
print("✅ Section plotted.")


## 6 · Instantaneous Amplitude (Envelope)

### Theory
The **analytic signal** of a real seismic trace $x(t)$ is:

$$\tilde{x}(t) = x(t) + j\,\hat{x}(t)$$

where $\hat{x}(t)$ is the **Hilbert transform** of $x(t)$.

From the analytic signal we derive three *instantaneous attributes*:

| Attribute | Formula | Physical meaning |
|-----------|---------|-----------------|
| **Envelope** (Inst. Amplitude) | $A(t)=|\tilde{x}(t)|$ | Reflection strength; bright-spot indicator |
| **Inst. Phase** | $\phi(t)=\angle\tilde{x}(t)$ | Continuity & lateral correlation |
| **Inst. Frequency** | $f(t)=\frac{1}{2\pi}\frac{d\phi}{dt}$ | Frequency shadows below gas sands |

The Hilbert transform is computed efficiently via FFT in `scipy.signal.hilbert`.


In [ ]:
print("Computing analytic signal (Hilbert transform) …  ", end="", flush=True)
t0 = time.time()

analytic   = hilbert(data, axis=0)          # shape (n_samples, n_traces)
env        = np.abs(analytic)               # instantaneous amplitude / envelope
phase_rad  = np.angle(analytic)             # instantaneous phase  (radians, -π … π)

# Instantaneous frequency: derivative of unwrapped phase / (2π dt)
phase_unwr = np.unwrap(phase_rad, axis=0)
inst_freq  = np.diff(phase_unwr, axis=0) / (2 * np.pi * dt_ms / 1000)   # Hz
# Pad last row so shape matches
inst_freq  = np.vstack([inst_freq, inst_freq[-1:, :]])

# Clip to physical range
inst_freq  = np.clip(inst_freq, 0, nyquist)

print(f"done  ({time.time()-t0:.1f} s)")
print(f"  env       shape {env.shape}  |  range [{env.min():.4f}, {env.max():.4f}]")
print(f"  phase_rad shape {phase_rad.shape}  |  range [{phase_rad.min():.2f}, {phase_rad.max():.2f}] rad")
print(f"  inst_freq shape {inst_freq.shape}  |  range [{inst_freq.min():.1f}, {inst_freq.max():.1f}] Hz")


### 6a · Envelope Section Plot

In [ ]:
sec_env = env[:, idx]

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(sec_env, aspect='auto', cmap=CMAP_AMP,
               vmin=0, vmax=np.percentile(sec_env, 98),
               extent=[idx[0], idx[-1], time_axis[-1], time_axis[0]])
plt.colorbar(im, ax=ax, label='Envelope (Reflection Strength)', shrink=0.85)
ax.set_xlabel("Trace index"); ax.set_ylabel("TWT (ms)")
ax.set_title("Instantaneous Amplitude (Envelope) — Reflection Strength Section")
plt.tight_layout(); plt.show()


## 7 · Instantaneous Phase & Instantaneous Frequency

### Theory — Phase
**Instantaneous phase** is independent of amplitude. It reveals:
- **Lateral continuity** of reflectors (constant phase = continuous layer)
- **Unconformities** and **faults** → phase discontinuities
- **Wavelet character** — helps identify zero-phase vs minimum-phase data

### Theory — Instantaneous Frequency
**Instantaneous frequency** responds to:
- **Gas accumulations** → frequency shadow (low-freq anomaly below reservoir)
- **Thin beds** → tuning effects cause frequency oscillations
- **Attenuation** → high-freq loss with depth

Values outside 0–Nyquist indicate cycle-skipping; we clip to physical bounds.


In [ ]:
sec_ph  = phase_rad[:, idx]
sec_if  = inst_freq[:, idx]

fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

im1 = axes[0].imshow(sec_ph, aspect='auto', cmap=CMAP_PHASE,
                     vmin=-np.pi, vmax=np.pi,
                     extent=[idx[0], idx[-1], time_axis[-1], time_axis[0]])
plt.colorbar(im1, ax=axes[0], label='Phase (radians)', shrink=0.85)
axes[0].set_xlabel("Trace index"); axes[0].set_ylabel("TWT (ms)")
axes[0].set_title("Instantaneous Phase")

im2 = axes[1].imshow(sec_if, aspect='auto', cmap=CMAP_FREQ,
                     vmin=0, vmax=nyquist,
                     extent=[idx[0], idx[-1], time_axis[-1], time_axis[0]])
plt.colorbar(im2, ax=axes[1], label='Frequency (Hz)', shrink=0.85)
axes[1].set_xlabel("Trace index")
axes[1].set_title("Instantaneous Frequency")

plt.suptitle("Phase & Frequency Attributes", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()


## 8 · Spectral (Frequency-Domain) Analysis

### Theory
The **power spectral density (PSD)** of seismic data shows:
- The **dominant frequency** of the source wavelet (peak of PSD)
- **Bandwidth** (3 dB roll-off frequencies)  
- **Low-frequency noise** (< 5 Hz) → swell / ground roll contamination  
- **High-frequency roll-off** → attenuation or anti-alias filter

We use Welch's method (averaged periodograms) for a stable estimate.  
We also plot a 2-D **f-k spectrum** (frequency vs trace-index), which
reveals coherent noise (linear events → vertical stripes in f-k space).


In [ ]:
# ── Average PSD across all traces (Welch) ─────────────────────────────────────
NPERSEG = min(N_SAMPLES, 64)
freqs_w, psd = welch(data.T, fs=fs, nperseg=NPERSEG, axis=1)
mean_psd = psd.mean(axis=0)

dom_freq = freqs_w[np.argmax(mean_psd)]
print(f"Dominant frequency (Welch): {dom_freq:.1f} Hz")

# ── f-k spectrum (FFT along both axes of sub-section) ─────────────────────────
NTRACE_FK = 512
idx_fk = np.linspace(0, N_TRACES-1, NTRACE_FK, dtype=int)
sec_fk = data[:, idx_fk]
FK = np.abs(np.fft.fftshift(np.fft.fft2(sec_fk)))
fk_freqs = np.fft.fftshift(np.fft.fftfreq(N_SAMPLES, d=dt_ms/1000))
fk_k     = np.fft.fftshift(np.fft.fftfreq(NTRACE_FK))

fig = plt.figure(figsize=(18, 6))
gs  = GridSpec(1, 3, figure=fig, wspace=0.35)

# PSD
ax1 = fig.add_subplot(gs[0])
ax1.semilogy(freqs_w, mean_psd, color='steelblue', lw=1.5)
ax1.axvline(dom_freq, color='red', ls='--', lw=1.5, label=f'Peak {dom_freq:.0f} Hz')
ax1.set_xlabel("Frequency (Hz)"); ax1.set_ylabel("Power spectral density")
ax1.set_title("Mean Power Spectral Density
(Welch, averaged over all traces)")
ax1.set_xlim(0, nyquist); ax1.legend(); ax1.grid(alpha=0.3)

# Per-trace spectrograms (one representative trace)
TR_SPEC = N_TRACES // 2
f_sg, t_sg, Sxx = spectrogram(data[:, TR_SPEC], fs=fs, nperseg=NPERSEG)
ax2 = fig.add_subplot(gs[1])
im2 = ax2.pcolormesh(t_sg*1000, f_sg, 10*np.log10(Sxx+1e-20), cmap='inferno', shading='auto')
plt.colorbar(im2, ax=ax2, label='dB')
ax2.set_xlabel("TWT (ms)"); ax2.set_ylabel("Frequency (Hz)")
ax2.set_title(f"Spectrogram — Trace {TR_SPEC}
(time-frequency energy map)")

# f-k
ax3 = fig.add_subplot(gs[2])
im3 = ax3.imshow(np.log1p(FK), aspect='auto', cmap='hot',
                 extent=[fk_k[0], fk_k[-1], fk_freqs[-1], fk_freqs[0]])
ax3.set_xlim(-0.5, 0.5)
ax3.set_ylim(0, nyquist)
plt.colorbar(im3, ax=ax3, label='log(1+|FK|)')
ax3.set_xlabel("Wavenumber (cycle/trace)"); ax3.set_ylabel("Frequency (Hz)")
ax3.set_title("f-k Spectrum
(noise identification)")

plt.suptitle("Spectral Analysis", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()


## 9 · Semblance / Coherence

### Theory
**Semblance** (normalised cross-energy) measures **lateral waveform similarity** across neighbouring traces.

$$S(t, x) = \frac{\left(\sum_{i=x-h}^{x+h} u_i(t)\right)^2}{n_w \sum_{i=x-h}^{x+h} u_i^2(t)}$$

where $n_w$ is the number of traces in the analysis window, $h$ is the half-aperture.

| Value | Meaning |
|-------|---------|
| $S \approx 1$ | Coherent reflector |
| $S \approx 0$ | Incoherent / chaotic (fault zone, gas cloud, noise) |

Semblance is the precursor to modern **coherence / similarity** attributes used  
for fault and channel detection.


In [ ]:
print("Computing semblance …  ", end="", flush=True)
t0 = time.time()

HALF_APT = 2          # half-aperture in traces
WIN_S    = 5          # vertical smoothing window (samples)

# Work on a representative sub-section to avoid memory issues
STRIDE = max(1, N_TRACES // 3000)
d_sub  = data[:, ::STRIDE]           # (n_samples, ~3000)
nt_sub = d_sub.shape[1]

semblance = np.zeros_like(d_sub)
for ti in range(HALF_APT, nt_sub - HALF_APT):
    patch   = d_sub[:, ti-HALF_APT : ti+HALF_APT+1]   # (ns, nw)
    numer   = patch.sum(axis=1) ** 2
    denom   = (patch**2).sum(axis=1) * (2*HALF_APT+1)
    semblance[:, ti] = numer / (denom + 1e-12)

semblance = uniform_filter1d(semblance, size=WIN_S, axis=0)

print(f"done  ({time.time()-t0:.1f} s)")

trace_idx_sub = np.arange(nt_sub) * STRIDE

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(semblance, aspect='auto', cmap='RdYlGn',
               vmin=0, vmax=1,
               extent=[trace_idx_sub[0], trace_idx_sub[-1],
                       time_axis[-1], time_axis[0]])
plt.colorbar(im, ax=ax, label='Semblance (0=incoherent, 1=coherent)', shrink=0.85)
ax.set_xlabel("Trace index"); ax.set_ylabel("TWT (ms)")
ax.set_title(f"Semblance (Coherence)  |  aperture=±{HALF_APT} traces")
plt.tight_layout(); plt.show()


## 10 · RMS Amplitude (Sliding Window)

### Theory
**RMS amplitude** computed in a sliding time window:

$$A_{\text{RMS}}(t,x) = \sqrt{\frac{1}{W}\sum_{k=t-W/2}^{t+W/2} u^2(k,x)}$$

This is the most widely used **hydrocarbon indicator** attribute:
- **Bright spots** (high RMS) over structural highs → possible gas sands  
- **Flat spots** (horizontal high-RMS stripe) → fluid contacts  
- **Dim spots** → hard over soft (e.g. tight carbonate on shale)

Window length $W$ is typically 20–50 ms (equivalent to ~1–2 dominant wavelengths).


In [ ]:
WIN_MS   = 20.0                          # window length in ms
WIN_SAMP = max(1, int(WIN_MS / dt_ms))  # samples

print(f"RMS window: {WIN_MS} ms  =  {WIN_SAMP} samples")
print("Computing RMS amplitude …  ", end="", flush=True)
t0 = time.time()

# Efficient sliding RMS via cumulative sum of squares
d2     = data ** 2
cs     = np.cumsum(d2, axis=0)
pad    = np.zeros((1, N_TRACES))
cs_pad = np.vstack([pad, cs])

half_w = WIN_SAMP // 2
idx_lo = np.maximum(0,          np.arange(N_SAMPLES) - half_w)
idx_hi = np.minimum(N_SAMPLES,  np.arange(N_SAMPLES) + half_w + 1)

rms_amp = np.zeros_like(data)
for t in range(N_SAMPLES):
    window_sum   = cs_pad[idx_hi[t]] - cs_pad[idx_lo[t]]
    window_len   = idx_hi[t] - idx_lo[t]
    rms_amp[t,:] = np.sqrt(window_sum / window_len)

print(f"done  ({time.time()-t0:.1f} s)")

sec_rms = rms_amp[:, idx]

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(sec_rms, aspect='auto', cmap='hot_r',
               vmin=0, vmax=np.percentile(sec_rms, 98),
               extent=[idx[0], idx[-1], time_axis[-1], time_axis[0]])
plt.colorbar(im, ax=ax, label=f'RMS Amplitude  ({WIN_MS:.0f} ms window)', shrink=0.85)
ax.set_xlabel("Trace index"); ax.set_ylabel("TWT (ms)")
ax.set_title(f"Sliding-Window RMS Amplitude  ({WIN_MS} ms)")
plt.tight_layout(); plt.show()


## 11 · Acoustic Impedance Proxy & Reflection Coefficient

### Theory
The **reflection coefficient** at an interface is:

$$RC = \frac{Z_2 - Z_1}{Z_2 + Z_1}$$

where $Z = \rho V$ is acoustic impedance.  
Seismic traces are band-limited **reflectivity estimates**. We can approximate the
impedance log by integrating the trace (recursive inversion):

$$Z(t) \approx Z_0 \exp\left(2\int_0^t RC(\tau)\,d\tau\right)$$

This is sometimes called **coloured inversion** or **recursive inversion**.  
It preserves relative impedance contrasts but not absolute values.

The derivative (first difference) of the trace approximates the **RC series**:

$$RC(t) \approx \frac{d}{dt}\left[\ln Z(t)\right] \approx \frac{x(t+\Delta t) - x(t)}{\Delta t}$$


In [ ]:
# ── Recursive integration → pseudo-impedance ──────────────────────────────────
print("Computing acoustic impedance proxy …  ", end="", flush=True)
t0 = time.time()

# High-pass filter data first (remove DC drift before integration)
b_hp, a_hp = butter(4, 5/(fs/2), btype='high')
data_hp     = filtfilt(b_hp, a_hp, data, axis=0)

# Integrate → pseudo-impedance (cumulative sum ≈ ∫)
pseudo_imp  = np.cumsum(data_hp, axis=0)
# Normalise each trace
norms       = np.percentile(np.abs(pseudo_imp), 95, axis=0) + 1e-10
pseudo_imp  = pseudo_imp / norms

# ── Reflection coefficient (first derivative of data) ─────────────────────────
rc_series   = np.diff(data, axis=0, prepend=data[:1,:])

print(f"done  ({time.time()-t0:.1f} s)")

sec_ai = pseudo_imp[:, idx]
sec_rc = rc_series[:, idx]
clip_rc = 2.0 * sec_rc.std()

fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

im1 = axes[0].imshow(sec_ai, aspect='auto', cmap='RdBu_r',
                     vmin=-1, vmax=1,
                     extent=[idx[0], idx[-1], time_axis[-1], time_axis[0]])
plt.colorbar(im1, ax=axes[0], label='Normalised Pseudo-Impedance', shrink=0.85)
axes[0].set_xlabel("Trace index"); axes[0].set_ylabel("TWT (ms)")
axes[0].set_title("Acoustic Impedance Proxy
(Band-limited Recursive Inversion)")

im2 = axes[1].imshow(sec_rc, aspect='auto', cmap=CMAP_SEIS,
                     vmin=-clip_rc, vmax=clip_rc,
                     extent=[idx[0], idx[-1], time_axis[-1], time_axis[0]])
plt.colorbar(im2, ax=axes[1], label='RC (arbitrary)', shrink=0.85)
axes[1].set_xlabel("Trace index")
axes[1].set_title("Reflection Coefficient Series
(first-derivative approximation)")

plt.suptitle("Impedance & Reflection Coefficient", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()


## 12 · Sweetness Attribute

### Theory
**Sweetness** is a composite attribute defined as:

$$\text{Sweetness}(t,x) = \frac{A(t,x)}{\sqrt{f_i(t,x)}}$$

where $A$ is the envelope and $f_i$ is the instantaneous frequency.

**Physical basis**: gas sands produce both a **bright amplitude** (high $A$) and a  
**low instantaneous frequency** (frequency shadow). Sweetness amplifies both effects
simultaneously, making it a powerful **direct hydrocarbon indicator (DHI)**.

Values are high where reflections are **strong AND low-frequency** — the hallmark
of gas-saturated sands.


In [ ]:
print("Computing Sweetness …  ", end="", flush=True)
t0 = time.time()

# Avoid division by zero; clip to minimum 1 Hz
f_safe   = np.clip(inst_freq, 1.0, None)
sweetness = env / np.sqrt(f_safe)

print(f"done  ({time.time()-t0:.1f} s)")

sec_sw = sweetness[:, idx]

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(sec_sw, aspect='auto', cmap='YlOrRd',
               vmin=0, vmax=np.percentile(sec_sw, 97),
               extent=[idx[0], idx[-1], time_axis[-1], time_axis[0]])
plt.colorbar(im, ax=ax, label='Sweetness  (Envelope / √Inst.Freq)', shrink=0.85)
ax.set_xlabel("Trace index"); ax.set_ylabel("TWT (ms)")
ax.set_title("Sweetness  —  Direct Hydrocarbon Indicator (DHI)")
plt.tight_layout(); plt.show()


## 13 · Dominant Frequency Map

### Theory
The **dominant frequency** per sample is estimated as the **centroid** of the  
instantaneous frequency, smoothed over a short time gate:

$$f_{\text{dom}}(t,x) = \text{smooth}\left[f_i(t,x)\right]$$

Plotted as a 2-D section it reveals:
- **Frequency shadows** below gas (low-frequency anomalies)  
- **Thin-bed tuning** (high-freq oscillations)  
- **Acquisition footprint** (systematic striping in the trace direction)


In [ ]:
# Smooth instantaneous frequency with a short window
SMOOTH_WIN = 9          # samples
dom_freq_map = uniform_filter1d(inst_freq, size=SMOOTH_WIN, axis=0)
dom_freq_map = np.clip(dom_freq_map, 0, nyquist)

sec_df = dom_freq_map[:, idx]

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(sec_df, aspect='auto', cmap=CMAP_FREQ,
               vmin=0, vmax=nyquist,
               extent=[idx[0], idx[-1], time_axis[-1], time_axis[0]])
plt.colorbar(im, ax=ax, label='Dominant Frequency (Hz)', shrink=0.85)
ax.set_xlabel("Trace index"); ax.set_ylabel("TWT (ms)")
ax.set_title(f"Dominant Frequency Map  (smoothing window = {SMOOTH_WIN} samples)")
plt.tight_layout(); plt.show()


## 14 · Full Attribute Gallery

### Summary of Attributes Computed

| # | Attribute | Domain | Primary Use |
|---|-----------|--------|-------------|
| 1 | Seismic amplitude | Time | Structural interpretation |
| 2 | Envelope (Inst. Amplitude) | Time | Reflection strength, bright spots |
| 3 | Instantaneous Phase | Time | Continuity, faults, unconformities |
| 4 | Instantaneous Frequency | Time | Gas shadows, attenuation |
| 5 | RMS Amplitude | Time | DHI, flat spots |
| 6 | Semblance | Time | Fault & fracture detection |
| 7 | Acoustic Impedance Proxy | Time | Lithology, fluid |
| 8 | Reflection Coefficient | Time | Impedance contrasts |
| 9 | Sweetness | Time | Gas sand DHI |
| 10 | Dominant Frequency | Time-Freq | Attenuation, thin beds |


In [ ]:
attrs = {
    "Seismic Amplitude"      : (data[:,     idx], CMAP_SEIS,  'Amplitude',       True,  2.0 ),
    "Envelope"               : (env[:,      idx], CMAP_AMP,   'Envelope',        False, None),
    "Inst. Phase"            : (phase_rad[:,idx], CMAP_PHASE, 'Phase (rad)',     False, None),
    "Inst. Frequency"        : (inst_freq[:,idx], CMAP_FREQ,  'Frequency (Hz)',  False, None),
    "RMS Amplitude"          : (rms_amp[:,  idx], 'hot_r',    'RMS Amplitude',   False, None),
    "Semblance"              : (semblance[:,  np.linspace(0,semblance.shape[1]-1, DISP_TRACES,dtype=int)],
                                'RdYlGn', 'Semblance', False, None),
    "Pseudo-Impedance"       : (pseudo_imp[:,idx],'RdBu_r',   'Norm. AI',        False, None),
    "Reflect. Coefficient"   : (rc_series[:,idx], CMAP_SEIS,  'RC',              True,  2.0 ),
    "Sweetness"              : (sweetness[:,idx],  'YlOrRd',   'Sweetness',       False, None),
    "Dominant Frequency"     : (dom_freq_map[:,idx],CMAP_FREQ,'Dom. Freq (Hz)',  False, None),
}

NROWS, NCOLS = 2, 5
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(24, 10), sharey=True)
axes = axes.ravel()

for k, (title, (arr, cmap, label, sym, nsig)) in enumerate(attrs.items()):
    ax = axes[k]
    if sym:
        vmax = nsig * arr.std()
        vmin = -vmax
    else:
        vmin = np.percentile(arr, 2)
        vmax = np.percentile(arr, 98)

    # match trace extent
    xlo = idx[0]; xhi = idx[-1]
    ax.imshow(arr, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax,
              extent=[xlo, xhi, time_axis[-1], time_axis[0]])
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xlabel("Trace", fontsize=7)
    if k % NCOLS == 0:
        ax.set_ylabel("TWT (ms)", fontsize=7)
    ax.tick_params(labelsize=6)

plt.suptitle("Complete Seismic Attribute Gallery", fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("seismic_attribute_gallery.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gallery saved as  seismic_attribute_gallery.png")


## Appendix — Export Attributes to NumPy / CSV

Run the cell below to save all computed attributes as `.npy` files for downstream use.


In [ ]:
SAVE = True          # set False to skip

if SAVE:
    np.save("attr_amplitude.npy",    data)
    np.save("attr_envelope.npy",     env)
    np.save("attr_phase.npy",        phase_rad)
    np.save("attr_inst_freq.npy",    inst_freq)
    np.save("attr_rms_amp.npy",      rms_amp)
    np.save("attr_semblance.npy",    semblance)
    np.save("attr_pseudo_imp.npy",   pseudo_imp)
    np.save("attr_rc.npy",           rc_series)
    np.save("attr_sweetness.npy",    sweetness)
    np.save("attr_dom_freq.npy",     dom_freq_map)
    print("✅ All attributes saved as .npy files in the current directory.")
else:
    print("Skipped (SAVE=False).")
